# 03 — Graph-Based Protocol Comparison & Cypher Queries (from scratch, offline)

Companion notebook to `../04-document-comparison-with-graphdb-and-cypher.md`.

This notebook builds a small **in-memory graph** (plain Python dicts/adjacency lists -- no Neo4j
instance required) representing a global clinical trial Protocol and two country-specific
variants (Germany, Japan), implements a **graph-diff function** to find what each country adds
or modifies (including *downstream* effects via requirement-to-requirement references), and
pairs each piece of Python logic with the equivalent **Cypher** query as a markdown reference
(shown as syntax, not executed -- there's no live Neo4j instance here, but the mapping from
Python traversal to Cypher pattern is 1:1).

Fully offline. Uses only the Python standard library; an optional `networkx` section at the end
runs only if `networkx` happens to be installed, and is skipped gracefully otherwise.

## 1. Modeling the global Protocol as requirement nodes

A handful of Protocol requirements, each tagged with a `type` (eligibility, lab, dosing,
safety). One requirement (`REQ-ELIG-02`, an eligibility criterion) explicitly **references**
another (`REQ-LAB-01`, a lab-reference-range requirement) -- this dependency edge is exactly the
kind of relationship a flat text diff has no way to represent.

In [1]:
GLOBAL_REQUIREMENTS = {
    "REQ-ELIG-01": {
        "type": "eligibility",
        "text": "Adults aged 18 to 75 years, inclusive, at the time of screening.",
    },
    "REQ-ELIG-02": {
        "type": "eligibility",
        "text": "Pre-bronchodilator FEV1 of 40-90% of predicted normal value at screening, per REQ-LAB-01.",
    },
    "REQ-LAB-01": {
        "type": "lab",
        "text": "Spirometry reference ranges per standard adult population norms.",
    },
    "REQ-DOSE-01": {
        "type": "dosing",
        "text": "100 mg subcutaneously every 4 weeks for participants weighing less than 60 kg.",
    },
    "REQ-SAFETY-01": {
        "type": "safety",
        "text": "Monitor for injection site reactions for 30 minutes following each dose.",
    },
}

# REQ-ELIG-02 depends on REQ-LAB-01 -- a relationship a flat text diff has no way to represent
REFERENCES = [
    ("REQ-ELIG-02", "REQ-LAB-01"),
]

print(f"{len(GLOBAL_REQUIREMENTS)} global requirement nodes, {len(REFERENCES)} REFERENCES edge(s)")

5 global requirement nodes, 1 REFERENCES edge(s)


## 2. Country-specific variants: `MODIFIES` and `ADDS` edges

Germany changes the lab reference-range requirement (`REQ-LAB-01`) to use a national society's
norms instead of generic adult norms, and adds a wholly new consent-disclosure requirement.
Japan changes the dosing requirement to a different dose per local regulatory guidance. Each
variant is modeled as a node connected to the requirement(s) it touches -- never as a second,
separate copy of the whole Protocol.

In [2]:
COUNTRY_VARIANTS = {
    "Germany": {
        "MODIFIES": {
            "REQ-LAB-01": (
                "Spirometry reference ranges per German national pulmonology society (DGP) "
                "norms, not generic adult norms."
            ),
        },
        "ADDS": {
            "REQ-DE-CONSENT-01": {
                "type": "consent",
                "text": (
                    "Additional consent disclosure required under German federal data "
                    "protection law (BDSG) for genetic sub-study samples."
                ),
            },
        },
    },
    "Japan": {
        "MODIFIES": {
            "REQ-DOSE-01": (
                "80 mg subcutaneously every 4 weeks for participants weighing less than 60 kg, "
                "per PMDA guidance."
            ),
        },
        "ADDS": {},
    },
}

def build_graph():
    """Assemble one combined graph: global requirement nodes + country-variant nodes,
    connected by REFERENCES (requirement -> requirement) and MODIFIES/ADDS (variant -> requirement)."""
    nodes = {"requirement": dict(GLOBAL_REQUIREMENTS), "country_variant": {}}
    edges = {"REFERENCES": list(REFERENCES), "MODIFIES": [], "ADDS": []}

    for country, changes in COUNTRY_VARIANTS.items():
        variant_id = f"VARIANT-{country.upper()}"
        nodes["country_variant"][variant_id] = {"country": country}
        for req_id, new_text in changes.get("MODIFIES", {}).items():
            edges["MODIFIES"].append((variant_id, req_id, new_text))
        for req_id, req_data in changes.get("ADDS", {}).items():
            nodes["requirement"][req_id] = req_data  # new requirement introduced by this country
            edges["ADDS"].append((variant_id, req_id))
    return nodes, edges

nodes, edges = build_graph()
print("Requirement nodes:", len(nodes["requirement"]))
print("Country variant nodes:", list(nodes["country_variant"]))
print("MODIFIES edges:", edges["MODIFIES"])
print("ADDS edges:", edges["ADDS"])

Requirement nodes: 6
Country variant nodes: ['VARIANT-GERMANY', 'VARIANT-JAPAN']
MODIFIES edges: [('VARIANT-GERMANY', 'REQ-LAB-01', 'Spirometry reference ranges per German national pulmonology society (DGP) norms, not generic adult norms.'), ('VARIANT-JAPAN', 'REQ-DOSE-01', '80 mg subcutaneously every 4 weeks for participants weighing less than 60 kg, per PMDA guidance.')]
ADDS edges: [('VARIANT-GERMANY', 'REQ-DE-CONSENT-01')]


**Equivalent Cypher schema** (reference only -- not executed, since no live Neo4j
instance is available in this offline notebook):

```cypher
CREATE (:Requirement {id: "REQ-LAB-01", type: "lab", text: "Spirometry reference ranges per standard adult population norms."})
CREATE (:Requirement {id: "REQ-ELIG-02", type: "eligibility", text: "..."})
CREATE (:CountryVariant {id: "VARIANT-GERMANY", country: "Germany"})

MATCH (a:Requirement {id: "REQ-ELIG-02"}), (b:Requirement {id: "REQ-LAB-01"})
CREATE (a)-[:REFERENCES]->(b)

MATCH (c:CountryVariant {id: "VARIANT-GERMANY"}), (r:Requirement {id: "REQ-LAB-01"})
CREATE (c)-[:MODIFIES {new_text: "Spirometry reference ranges per German national pulmonology society (DGP) norms..."}]->(r)
```

## 3. Graph-diff function: what changed for a given country?

The core operation: given a country, find every requirement it directly modifies or adds, *and*
follow `REFERENCES` edges to find requirements that depend on something changed -- the compound,
easy-to-miss difference a flat text diff cannot surface.

In [3]:
def diff_for_country(nodes, edges, country: str):
    """Requirement-level diff for one country: direct changes plus downstream-affected
    requirements (anything that REFERENCES a requirement this country changed)."""
    variant_id = f"VARIANT-{country.upper()}"

    modified = [(req_id, new_text) for (vid, req_id, new_text) in edges["MODIFIES"] if vid == variant_id]
    added = [req_id for (vid, req_id) in edges["ADDS"] if vid == variant_id]

    changed_ids = {req_id for req_id, _ in modified} | set(added)
    downstream = [
        (src, dst) for (src, dst) in edges["REFERENCES"]
        if dst in changed_ids and src not in changed_ids
    ]

    return {"modified": modified, "added": added, "downstream_affected": downstream}


de_diff = diff_for_country(nodes, edges, "Germany")
print("Germany diff:")
print("  modified:          ", de_diff["modified"])
print("  added:              ", de_diff["added"])
print("  downstream_affected:", de_diff["downstream_affected"])

assert de_diff["added"] == ["REQ-DE-CONSENT-01"]
assert de_diff["downstream_affected"] == [("REQ-ELIG-02", "REQ-LAB-01")]
print("\nOK: Germany diff finds both the direct lab-requirement modification AND the downstream "
      "eligibility criterion (REQ-ELIG-02) that references it -- a compound difference a flat "
      "text diff between two documents would not surface as a single, connected finding.")

Germany diff:
  modified:           [('REQ-LAB-01', 'Spirometry reference ranges per German national pulmonology society (DGP) norms, not generic adult norms.')]
  added:               ['REQ-DE-CONSENT-01']
  downstream_affected: [('REQ-ELIG-02', 'REQ-LAB-01')]

OK: Germany diff finds both the direct lab-requirement modification AND the downstream eligibility criterion (REQ-ELIG-02) that references it -- a compound difference a flat text diff between two documents would not surface as a single, connected finding.


**Equivalent Cypher** — direct changes for a country (maps to `modified` + `added` above):

```cypher
MATCH (c:CountryVariant {country: "Germany"})-[r:MODIFIES|ADDS]->(req:Requirement)
RETURN req.id, req.type, req.text, type(r) AS change_type
```

**Equivalent Cypher** — downstream-affected requirements (maps to `downstream_affected` above,
the query a flat text diff cannot express):

```cypher
MATCH (c:CountryVariant {country: "Germany"})-[:MODIFIES]->(changed:Requirement)
MATCH (dependent:Requirement)-[:REFERENCES]->(changed)
RETURN dependent.id, dependent.text, changed.id AS depends_on_changed_requirement
```

In [4]:
jp_diff = diff_for_country(nodes, edges, "Japan")
print("Japan diff:")
print("  modified:          ", jp_diff["modified"])
print("  added:              ", jp_diff["added"])
print("  downstream_affected:", jp_diff["downstream_affected"])

assert jp_diff["modified"] == [
    ("REQ-DOSE-01", "80 mg subcutaneously every 4 weeks for participants weighing less than 60 kg, per PMDA guidance.")
]
assert jp_diff["downstream_affected"] == []
print("\nOK: Japan only changes a dosing requirement with no downstream dependents in this graph, "
      "so downstream_affected is correctly empty.")

Japan diff:
  modified:           [('REQ-DOSE-01', '80 mg subcutaneously every 4 weeks for participants weighing less than 60 kg, per PMDA guidance.')]
  added:               []
  downstream_affected: []

OK: Japan only changes a dosing requirement with no downstream dependents in this graph, so downstream_affected is correctly empty.


## 4. Comparing two countries against each other

Because both countries' variant nodes point at the *same* underlying requirement nodes, finding
requirements that multiple countries touch is a simple set intersection -- this generalizes the
comparison from "global vs. one country" to "country vs. country" almost for free.

In [5]:
def changed_requirement_ids(edges, country: str):
    variant_id = f"VARIANT-{country.upper()}"
    ids = {req_id for (vid, req_id, _) in edges["MODIFIES"] if vid == variant_id}
    ids |= {req_id for (vid, req_id) in edges["ADDS"] if vid == variant_id}
    return ids

def compare_countries(edges, country_a: str, country_b: str):
    return sorted(changed_requirement_ids(edges, country_a) & changed_requirement_ids(edges, country_b))

shared = compare_countries(edges, "Germany", "Japan")
print("Requirements changed by BOTH Germany and Japan:", shared)
assert shared == []  # expected: this synthetic example gives them non-overlapping changes
print("OK: no overlap here (Germany changed a lab requirement, Japan changed a dosing "
      "requirement) -- with real data this query is how you would spot two countries diverging "
      "on the same underlying requirement.")

Requirements changed by BOTH Germany and Japan: []
OK: no overlap here (Germany changed a lab requirement, Japan changed a dosing requirement) -- with real data this query is how you would spot two countries diverging on the same underlying requirement.


**Equivalent Cypher**:

```cypher
MATCH (c1:CountryVariant {country: "Germany"})-[:MODIFIES|ADDS]->(req:Requirement)
MATCH (c2:CountryVariant {country: "Japan"})-[:MODIFIES|ADDS]->(req)
RETURN req.id, req.text
```

## 5. Optional: the same graph in `networkx` (only if installed)

`networkx` isn't required for anything above -- the diff logic is plain dict/list traversal. But
if it happens to be available, building the same structure as a `networkx.MultiDiGraph` is a
convenient way to get free graph utilities (shortest path, visualization, etc.) on top of the
same data. This cell is guarded and simply explains itself if `networkx` isn't present.

In [6]:
try:
    import networkx as nx

    G = nx.MultiDiGraph()
    for req_id, data in nodes["requirement"].items():
        G.add_node(req_id, kind="requirement", **data)
    for variant_id, data in nodes["country_variant"].items():
        G.add_node(variant_id, kind="country_variant", **data)
    for src, dst in edges["REFERENCES"]:
        G.add_edge(src, dst, relation="REFERENCES")
    for vid, req_id, new_text in edges["MODIFIES"]:
        G.add_edge(vid, req_id, relation="MODIFIES", new_text=new_text)
    for vid, req_id in edges["ADDS"]:
        G.add_edge(vid, req_id, relation="ADDS")

    print(f"networkx graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    # sanity check: does the graph-native traversal agree with our plain-Python diff?
    germany_targets = {
        v for _, v, d in G.out_edges("VARIANT-GERMANY", data=True)
        if d["relation"] in ("MODIFIES", "ADDS")
    }
    assert germany_targets == {"REQ-LAB-01", "REQ-DE-CONSENT-01"}
    print("OK: networkx traversal agrees with the plain-Python diff_for_country() result.")

except ImportError:
    print(
        "networkx is not installed in this environment -- that's fine, nothing above depends on "
        "it. The plain Python dict/adjacency-list version in Sections 1-4 is the complete, "
        "fully offline implementation; networkx (or a real Neo4j instance running Cypher) would "
        "simply be a more scalable/queryable backend for the exact same graph structure."
    )

networkx graph built: 8 nodes, 4 edges
OK: networkx traversal agrees with the plain-Python diff_for_country() result.


## 6. Tying it back

- The `diff_for_country` function above is the graph-native version of "Country Specific Protocol
  Comparison" -- it doesn't just find changed text, it finds changed *requirements*, tagged by
  type, and follows `REFERENCES` edges to surface compound, downstream effects a text diff would
  miss entirely.
- Every Python traversal in this notebook has a directly corresponding **Cypher** query shown
  alongside it -- in a real deployment, that Cypher would run against a live Neo4j (or similar)
  instance holding the actual Protocol graph, but the *logic* is identical to what you just ran
  in plain Python.
- This is the strongest possible answer to "why a graph instead of a diff": the value isn't the
  database technology, it's that requirements-as-nodes-with-relationships is a truthful model of
  what a Protocol actually is, and a flat two-document text diff simply has no way to represent
  that structure.